In [ ]:
%sql
DROP TABLE IF EXISTS workspace.gold_weather.kpi_agro_daily;

CREATE TABLE IF NOT EXISTS workspace.gold_weather.kpi_agro_daily (
  location_id BIGINT,
  date_key BIGINT,
  data_type STRING,
  frost_risk BOOLEAN,
  growing_degree_days DOUBLE,
  irrigation_deficit_mm DOUBLE
)

In [ ]:
# Umbral de helada (0 C) y temperatura base de GDD (10 C) son supuestos por
# defecto sin validar con el usuario, ver Proyecto/DECISIONS.md #6
kpi = spark.sql("""
    SELECT
        location_id,
        date_key,
        data_type,
        temp_min <= 0 AS frost_risk,
        greatest(0, temp_mean - 10) AS growing_degree_days,
        greatest(0, et0_fao_evapotranspiration - precipitation_sum) AS irrigation_deficit_mm
    FROM workspace.gold_weather.fact_weather_daily
""")

kpi.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_weather.kpi_agro_daily")